# Fine-tuning do MedGemma com QLoRA - Derm7pt
Aplica QLoRA no MedGemma 4B usando o dataset Derm7pt multiclasse com balanceamento + image augmentation.

In [ ]:
import os

REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q -e {REPO_DIR}
!pip install -q -U transformers accelerate peft trl bitsandbytes huggingface_hub

In [ ]:
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from melanoma_tcc.data.preprocessing import (
    Derm7ptDataset, GROUP_TO_LABEL, GROUP_TO_NAME, LABEL_TO_GROUP,
    balanced_subset_indices,
)
from melanoma_tcc.model.finetuning import load_model_for_finetuning, apply_lora, get_trainer
from melanoma_tcc.model.inference import predict, extract_multiclass_label
from melanoma_tcc.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7pt-multimodal/release_v0"
META_CSV = f"{DERM7PT_DIR}/meta/meta.csv"
IMAGES_DIR = f"{DERM7PT_DIR}/images"
TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"

print(f"Meta CSV existe: {os.path.exists(META_CSV)}")
print(f"Images dir existe: {os.path.exists(IMAGES_DIR)}")

In [ ]:
model, processor = load_model_for_finetuning(HF_TOKEN)
model = apply_lora(model)

In [ ]:
# Train: balanceado + augmentation
train_dataset = Derm7ptDataset(
    META_CSV, IMAGES_DIR, processor,
    indexes_csv=TRAIN_IDX,
    balance=True,
    target_per_class=80,
    max_oversample=4,
    augment=True,
    seed=42,
)

# Val: subset balanceado de 50 amostras (10 por classe) - economiza memoria/tempo
val_subset_idx = balanced_subset_indices(META_CSV, VAL_IDX, per_class=10, seed=42)
pd.DataFrame({"indexes": val_subset_idx}).to_csv("/kaggle/working/val_subset_idx.csv", index=False)
val_dataset = Derm7ptDataset(
    META_CSV, IMAGES_DIR, processor,
    indexes_csv="/kaggle/working/val_subset_idx.csv",
    balance=False,
    augment=False,
)

print(f"Train balanceado: {len(train_dataset)} amostras")
print(f"Val subset: {len(val_dataset)} amostras")

from collections import Counter
train_groups = [train_dataset[i]['group'] for i in range(len(train_dataset))]
print(f"\nTrain distribuicao: {Counter(train_groups)}")
val_groups = [val_dataset[i]['group'] for i in range(len(val_dataset))]
print(f"Val distribuicao: {Counter(val_groups)}")

sample = train_dataset[0]
print(f"\n=== Exemplo ===")
print(f"Prompt: {sample['prompt'][:120]}")
print(f"Resposta: {sample['answer']}")
print(f"Label: {sample['label']} ({sample['group']})")

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

In [ ]:
trainer = get_trainer(
    model, processor, train_dataset, val_dataset,
    output_dir="/kaggle/working/medgemma-derm7pt-v2"
)
trainer.train()

In [ ]:
trainer.save_model("/kaggle/working/medgemma-derm7pt-v2-final")
processor.save_pretrained("/kaggle/working/medgemma-derm7pt-v2-final")
print("Modelo salvo em /kaggle/working/medgemma-derm7pt-v2-final")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best metric (val loss): {trainer.state.best_metric}")

In [ ]:
model.eval()

test_dataset = Derm7ptDataset(META_CSV, IMAGES_DIR, processor, indexes_csv=TEST_IDX)
print(f"Test set: {len(test_dataset)} amostras")

test_groups = [test_dataset[i]['group'] for i in range(len(test_dataset))]
print(f"Distribuicao test (intocado): {Counter(test_groups)}")

In [ ]:
labels, predictions, responses = [], [], []
SAVE_EVERY = 20
CSV_PATH = "/kaggle/working/derm7pt_v2_predictions.csv"

def save_progress():
    df = pd.DataFrame({
        'true_label': labels,
        'pred_label': predictions,
        'true_group': [LABEL_TO_GROUP.get(l, "INVALID") for l in labels],
        'pred_group': [LABEL_TO_GROUP.get(p, "INVALID") for p in predictions],
        'response': responses,
    })
    df.to_csv(CSV_PATH, index=False)

for i in range(len(test_dataset)):
    sample = test_dataset[i]
    response = predict(model, processor, sample['image'], sample['prompt'])
    pred_label = extract_multiclass_label(response)
    labels.append(sample['label'])
    predictions.append(pred_label)
    responses.append(response)
    if i % SAVE_EVERY == 0:
        save_progress()
        print(f'[{i}/{len(test_dataset)}] label={sample["label"]} pred={pred_label} | {response[:80]} [SAVED]')

save_progress()
print(f'\nInferencia concluida! Salvo em {CSV_PATH}')

In [ ]:
print("=== Distribuicao das predicoes ===")
pred_groups = [LABEL_TO_GROUP.get(p, "INVALID") for p in predictions]
print(Counter(pred_groups))

print("\n=== Primeiras 5 respostas ===")
for i, r in enumerate(responses[:5]):
    print(f"[{labels[i]}] {r[:150]}")

In [ ]:
valid = [(l, p) for l, p in zip(labels, predictions) if p != -1]
invalid_count = len(labels) - len(valid)
print(f"Predicoes invalidas: {invalid_count}")

valid_labels, valid_preds = zip(*valid)
TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]
results = compute_metrics(list(valid_labels), list(valid_preds), target_names=TARGET_NAMES)
plot_confusion_matrix(
    list(valid_labels), list(valid_preds),
    save_path='/kaggle/working/derm7pt_v2_cm.png',
    target_names=TARGET_NAMES
)

In [ ]:
import shutil
shutil.make_archive(
    "/kaggle/working/medgemma-derm7pt-v2-final",
    "zip",
    "/kaggle/working/medgemma-derm7pt-v2-final"
)
print("ZIP do modelo criado em /kaggle/working/medgemma-derm7pt-v2-final.zip")
print("Baixa o ZIP + o CSV de predicoes ANTES de fechar o notebook!")